# **Baseline**
 A standard CNN processing the raw image to predict the scene label/embedding, ignoring the explicit graph structure.

CNN -> enbendigs -> contrastive loss

## Dataset

In [ ]:
import os
from PIL import Image
from torch.utils.data import Dataset

class SceneDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

        # mapping label → int
        self.label2idx = {l: i for i, l in enumerate(df['labels'].unique())}
        self.labels = df['labels'].map(self.label2idx).values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        img_path = os.path.join(self.image_dir, row['image_id'] + ".jpg")
        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        label = self.labels[idx]

        return img, label

## Balanced Batch Sampler

In [ ]:
import numpy as np
from torch.utils.data import Sampler

class BalancedBatchSampler(Sampler):
    """Genera batch finiti per epoch: n_classes x n_samples."""

    def __init__(self, labels, n_classes, n_samples):
        self.labels = np.array(labels)
        self.labels_set = np.unique(self.labels)
        self.label_to_indices = {label: np.where(self.labels == label)[0] for label in self.labels_set}

        self.n_classes = int(n_classes)
        self.n_samples = int(n_samples)

        if len(self.labels_set) < self.n_classes:
            raise ValueError(f"Not enough classes: {len(self.labels_set)} < n_classes={self.n_classes}")

    def __iter__(self):
        # IMPORTANTE: yield finito, altrimenti l'epoch non termina
        for _ in range(len(self)):
            classes = np.random.choice(self.labels_set, self.n_classes, replace=False)
            indices = []
            for c in classes:
                idxs = np.random.choice(self.label_to_indices[c], self.n_samples, replace=True)
                indices.extend(idxs.tolist())
            yield indices

    def __len__(self):
        return len(self.labels) // (self.n_classes * self.n_samples)

In [ ]:
from torchvision import transforms

# Con backbone ImageNet pre-trained, serve la normalizzazione ImageNet
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

 ## Dataloader

In [ ]:
from torch.utils.data import DataLoader

dataset = SceneDataset(df, "../data/images", transform)

sampler = BalancedBatchSampler(
    dataset.labels,
    n_classes=6,
    n_samples=4
)

loader = DataLoader(dataset, batch_sampler=sampler)

## CNN + Embedding 

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

class EmbeddingNet(nn.Module):
    def __init__(self, dim=128, pretrained=True):
        super().__init__()
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        base = models.resnet18(weights=weights)

        self.encoder = nn.Sequential(*list(base.children())[:-1])
        self.fc = nn.Linear(512, dim)

    def forward(self, x):
        x = self.encoder(x)
        x = x.flatten(1)
        x = self.fc(x)
        return F.normalize(x, dim=1)

## Contrastive loss

In [ ]:
import torch

def supervised_contrastive_loss(features, labels, temperature=0.1):
    """SupCon: esclude self-pairs e gestisce classi con 1 solo esempio nel batch."""
    device = features.device
    labels = labels.to(device)

    # (B,B) mask dei positivi (stessa classe)
    mask = labels[:, None].eq(labels[None, :]).float()

    logits = (features @ features.T) / temperature
    logits = logits - logits.max(dim=1, keepdim=True).values.detach()

    # escludi diagonale
    self_mask = torch.eye(features.size(0), device=device)
    exp_logits = torch.exp(logits) * (1.0 - self_mask)

    log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True) + 1e-9)

    pos_mask = mask * (1.0 - self_mask)
    denom = pos_mask.sum(dim=1).clamp_min(1.0)
    mean_log_prob_pos = (pos_mask * log_prob).sum(dim=1) / denom

    return -mean_log_prob_pos.mean()

## Training Loop

In [ ]:
import torch
from torch.optim import Adam

device = "cuda" if torch.cuda.is_available() else "cpu"

model = EmbeddingNet(dim=128, pretrained=True).to(device)
optimizer = Adam(model.parameters(), lr=1e-4)

epochs = 10
for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0.0
    n_batches = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        emb = model(images)
        loss = supervised_contrastive_loss(emb, labels, temperature=0.1)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        total_loss += float(loss.item())
        n_batches += 1

    print(f"Epoch {epoch}/{epochs} - loss: {total_loss / max(1, n_batches):.4f}")

## Retrival evaluation 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

@torch.no_grad()
def compute_embeddings(model, dataset, device, batch_size=64):
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    all_emb = []
    all_lbl = []
    all_imgs = []  # keep tensors for visualization

    for images, labels in loader:
        images = images.to(device)
        emb = model(images).detach().cpu()
        all_emb.append(emb)
        all_lbl.append(labels.detach().cpu())
        all_imgs.append(images.detach().cpu())

    emb = torch.cat(all_emb, dim=0)
    lbl = torch.cat(all_lbl, dim=0)
    imgs = torch.cat(all_imgs, dim=0)
    return emb, lbl, imgs


def retrieval_metrics_at_1(emb, labels):
    """1-NN retrieval within the same set (exclude self)."""
    # cosine similarity because embeddings are normalized
    sim = emb @ emb.T
    sim.fill_diagonal_(-1e9)
    nn_idx = sim.argmax(dim=1)
    pred = labels[nn_idx]

    y_true = labels.numpy()
    y_pred = pred.numpy()

    acc = float((y_true == y_pred).mean())

    # macro precision/recall
    num_classes = int(labels.max().item()) + 1
    eps = 1e-12
    precs, recs = [], []
    for c in range(num_classes):
        tp = np.sum((y_true == c) & (y_pred == c))
        fp = np.sum((y_true != c) & (y_pred == c))
        fn = np.sum((y_true == c) & (y_pred != c))
        precs.append(tp / (tp + fp + eps))
        recs.append(tp / (tp + fn + eps))

    return acc, float(np.mean(precs)), float(np.mean(recs)), nn_idx


def unnormalize_for_plot(img_tensor):
    """img_tensor: (3,H,W) normalized with ImageNet stats."""
    mean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
    std = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
    x = img_tensor * std + mean
    return x.clamp(0, 1)


def plot_retrieval(imgs, labels, sim, query_idx, topk=5, title=None):
    sim_q = sim[query_idx].clone()
    sim_q[query_idx] = -1e9
    top_idx = torch.topk(sim_q, k=topk).indices.tolist()

    ncols = topk + 1
    plt.figure(figsize=(3 * ncols, 3))

    # query
    plt.subplot(1, ncols, 1)
    q = unnormalize_for_plot(imgs[query_idx]).permute(1, 2, 0).numpy()
    plt.imshow(q)
    plt.axis('off')
    plt.title(f"Query\nlabel={int(labels[query_idx])}")

    # retrieved
    for j, ridx in enumerate(top_idx, start=2):
        plt.subplot(1, ncols, j)
        im = unnormalize_for_plot(imgs[ridx]).permute(1, 2, 0).numpy()
        plt.imshow(im)
        plt.axis('off')
        plt.title(f"Top{j-1}\nlabel={int(labels[ridx])}\nsim={float(sim[query_idx, ridx]):.2f}")

    if title:
        plt.suptitle(title)
    plt.tight_layout()
    plt.show()


# --- Run evaluation on the current `dataset`
emb, lbl, imgs = compute_embeddings(model, dataset, device=device, batch_size=64)

sim = emb @ emb.T
acc1, prec1, rec1, nn_idx = retrieval_metrics_at_1(emb, lbl)
print(f"Retrieval@1  acc={acc1:.4f}  precision_macro={prec1:.4f}  recall_macro={rec1:.4f}")

# Qualitative: show a random query and its top-K neighbors
query_idx = int(torch.randint(low=0, high=len(dataset), size=(1,)).item())
plot_retrieval(imgs, lbl, sim, query_idx=query_idx, topk=5, title="CNN baseline retrieval (cosine)")

# Optional: show a failure case if exists
wrong = (lbl[nn_idx] != lbl).nonzero().flatten()
if len(wrong) > 0:
    q_fail = int(wrong[0].item())
    plot_retrieval(imgs, lbl, sim, query_idx=q_fail, topk=5, title="Example failure case")